# AniData Lab : Rapport d'Audit Du Dataset

## Objectifs de cet audit

1. Explorer les 3 fichiers CSV sources (anime, ratings, synopsis)
2. Identifier les problèmes de qualité : valeurs manquantes, doublons, types incohérents, encodages
3. Produire des statistiques descriptives et un profiling automatisé
4. Documenter les anomalies et recommandations de nettoyage

# 1 Mise En Place et Chargement Des Données

In [23]:
import numpy as np
import pandas as pd

In [24]:
#Grand DF, pandas trinque beaucoup de colonnes
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [25]:
# Variablisation du repertoire source pour meilleur robustesse 
REP_SOURCE = "../data"

In [ ]:
animelist = pd.read_csv(f"{REP_SOURCE}/animelist.csv")
anime = pd.read_csv(f"{REP_SOURCE}/anime.csv", encoding='utf-8') #Verfication encodage des données
rating_complete = pd.read_csv(f"{REP_SOURCE}/rating_complete.csv")
anime_with_synopsis = pd.read_csv(f"{REP_SOURCE}/anime_with_synopsis.csv")

# 2 EDA, Nettoyage et Audit de animelist

In [27]:
animelist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109224747 entries, 0 to 109224746
Data columns (total 5 columns):
 #   Column            Dtype
---  ------            -----
 0   user_id           int64
 1   anime_id          int64
 2   rating            int64
 3   watching_status   int64
 4   watched_episodes  int64
dtypes: int64(5)
memory usage: 4.1 GB


In [28]:
animelist.isnull().sum()

user_id             0
anime_id            0
rating              0
watching_status     0
watched_episodes    0
dtype: int64

In [29]:
animelist.describe()

,user_id,anime_id,rating,watching_status,watched_episodes
count,1.092247e+08,1.092247e+08,1.092247e+08,1.092247e+08,1.092247e+08
mean,1.768098e+05,1.649590e+04,4.245717e+00,3.087289e+00,1.210818e+01
std,1.018487e+05,1.379737e+04,3.912888e+00,1.774407e+00,1.463155e+02
min,0.000000e+00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,8.849100e+04,3.194000e+03,0.000000e+00,2.000000e+00,0.000000e+00
50%,1.771420e+05,1.244500e+04,5.000000e+00,2.000000e+00,3.000000e+00
75%,2.651870e+05,3.083100e+04,8.000000e+00,6.000000e+00,1.200000e+01
max,3.534040e+05,4.849200e+04,1.000000e+01,5.500000e+01,6.553500e+04


In [30]:
animelist.head()

,user_id,anime_id,rating,watching_status,watched_episodes
0,0,67,9,1,1
1,0,6702,7,1,4
2,0,242,10,1,4
3,0,4898,0,1,1
4,0,21,10,1,0


In [31]:
nb_null_animelist = animelist.duplicated().sum()
print(nb_null_animelist)

1


In [32]:
animelist = animelist.drop_duplicates()
nb_null_animelist = animelist.duplicated().sum()
print(nb_null_animelist)

0


In [33]:
animelist[["user_id", "anime_id"]].duplicated().sum()

np.int64(0)

In [34]:
animelist["rating"].value_counts()

rating
0     46827034
8     15422150
7     14244633
9     10235934
6      7543377
10     7144392
5      4029645
4      1845854
3       905700
2       545339
1       480688
Name: count, dtype: int64

In [35]:
animelist["watching_status"].unique()

array([ 1,  2,  3,  4,  6,  0,  5, 33, 55])

In [36]:
animelist["watching_status"].value_counts()

watching_status
2     68089751
6     27938692
1      5228658
4      4266591
3      3700514
0          531
5            6
33           2
55           1
Name: count, dtype: int64

In [37]:
animelist["watching_status"] = animelist["watching_status"].replace({33: 3, 55: 5})

In [38]:
animelist["watching_status"].value_counts()

watching_status
2    68089751
6    27938692
1     5228658
4     4266591
3     3700516
0         531
5           7
Name: count, dtype: int64

In [39]:
animelist["watched_episodes"].value_counts()

watched_episodes
0        31436236
1        18649591
12       18012508
13        8150010
24        4145984
2         3565169
25        3455815
26        3322520
3         2103188
4         1842437
10        1619294
6         1521060
11        1434279
5         1010156
22         744959
7          641069
8          634506
51         517208
9          461086
14         375801
23         367032
50         311896
52         256587
15         248753
64         246929
37         219826
20         217161
39         196258
27         158770
220        140591
16         134547
43         102728
21          97155
148         91914
291         80475
49          78221
38          77593
175         77334
153         75632
47          74997
18          73664
366         67835
276         64096
500         62615
17          62423
75          58070
54          53911
19          52005
48          51529
46          51063
102         51020
60          50683
167         47166
103         44895
120        

In [40]:
animelist[animelist["watched_episodes"] > 500].value_counts()

user_id  anime_id  rating  watching_status  watched_episodes
1        21        9       1                958                 1
11       21        10      1                756                 1
14       21        8       4                689                 1
15       21        0       1                965                 1
30       21        8       3                705                 1
40       21        10      1                803                 1
43       21        9       1                692                 1
48       21        10      1                788                 1
50       21        10      1                825                 1
51       21        9       1                756                 1
55       21        10      1                580                 1
60       21        0       1                697                 1
66       2471      0       2                1787                1
71       21        0       1                697                 1
80       21    

In [41]:
animelist["watched_episodes"].unique()

array([   1,    4,    0, ..., 1266, 1039, 6688], shape=(1464,))

In [42]:
animelist["watched_episodes"].unique().max()

np.int64(65535)

In [43]:
animelist[animelist["watched_episodes"] == 65535]

,user_id,anime_id,rating,watching_status,watched_episodes
1040262,3546,1636,10,1,65535
1040263,3546,1896,0,1,65535
1572985,5411,34572,10,1,65535
1966788,6695,32977,6,1,65535
1966789,6695,34566,7,1,65535
1966791,6695,6149,8,1,65535
1966792,6695,966,7,1,65535
1966793,6695,235,6,1,65535
1966794,6695,8687,7,1,65535
1966796,6695,36838,6,1,65535


Données potentiellement abérentes. différents animes ont l'air  d'avoir le même nombre d'episodes mais certaines lignes sont marqué finis et d'autres encours avec le même nombre d'épisode vu? Un autre dataset sera necessaire pour corriger ce point de données et d'autres potentiel données abérentes.

In [44]:
animelist["watched_episodes"].unique().min()

np.int64(0)

In [45]:
animelist["watched_episodes"].describe()

count    1.092247e+08
mean     1.210818e+01
std      1.463155e+02
min      0.000000e+00
25%      0.000000e+00
50%      3.000000e+00
75%      1.200000e+01
max      6.553500e+04
Name: watched_episodes, dtype: float64

## Synthèse animelist

| Anomalie | Détail | Action |
|----------|--------|--------|
| 1 doublon exact | Ligne identique sur toutes les colonnes | Supprimé via `drop_duplicates()` |
| rating = 0 | 46.8M lignes (43% du dataset) | Valeur sentinelle = "pas encore noté", à distinguer des vraies notes |
| watching_status aberrants | 33 (2 lignes), 55 (1 ligne), 0 (531 lignes) | 33→3 et 55→5 corrigés. Status 0 = non documenté, à investiguer |
| watched_episodes = 65535 | 448 lignes, valeur = 2^16 - 1 | Overflow ou sentinelle, données aberrantes. Nécessite croisement avec anime.csv pour correction |

**Note :** Ce fichier fait 109M lignes (4.1 Go en mémoire). Il est mentionné comme optionnel dans le projet (challenge avancé). Le fichier principal pour les ratings est `rating_complete.csv`.

# 3 EDA, Nettoyage et Audit de anime

In [46]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   MAL_ID         17562 non-null  int64 
 1   Name           17562 non-null  object
 2   Score          17562 non-null  object
 3   Genres         17562 non-null  object
 4   English name   17562 non-null  object
 5   Japanese name  17562 non-null  object
 6   Type           17562 non-null  object
 7   Episodes       17562 non-null  object
 8   Aired          17562 non-null  object
 9   Premiered      17562 non-null  object
 10  Producers      17562 non-null  object
 11  Licensors      17562 non-null  object
 12  Studios        17562 non-null  object
 13  Source         17562 non-null  object
 14  Duration       17562 non-null  object
 15  Rating         17562 non-null  object
 16  Ranked         17562 non-null  object
 17  Popularity     17562 non-null  int64 
 18  Members        17562 non-n

In [47]:
anime.isnull().sum()

MAL_ID           0
Name             0
Score            0
Genres           0
English name     0
Japanese name    0
Type             0
Episodes         0
Aired            0
Premiered        0
Producers        0
Licensors        0
Studios          0
Source           0
Duration         0
Rating           0
Ranked           0
Popularity       0
Members          0
Favorites        0
Watching         0
Completed        0
On-Hold          0
Dropped          0
Plan to Watch    0
Score-10         0
Score-9          0
Score-8          0
Score-7          0
Score-6          0
Score-5          0
Score-4          0
Score-3          0
Score-2          0
Score-1          0
dtype: int64

In [48]:
anime.head()

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,Producers,Licensors,Studios,Source,Duration,Rating,Ranked,Popularity,Members,Favorites,Watching,Completed,On-Hold,Dropped,Plan to Watch,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,Bandai Visual,"Funimation, Bandai Entertainment",Sunrise,Original,24 min. per ep.,R - 17+ (violence & profanity),28.0,39,1251960,61971,105808,718161,71513,26678,329800,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,"Sunrise, Bandai Visual",Sony Pictures Entertainment,Bones,Original,1 hr. 55 min.,R - 17+ (violence & profanity),159.0,518,273145,1174,4143,208333,1935,770,57964,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,Victor Entertainment,"Funimation, Geneon Entertainment USA",Madhouse,Manga,24 min. per ep.,PG-13 - Teens 13 or older,266.0,201,558913,12944,29113,343492,25465,13925,146918,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,"TV Tokyo, Bandai Visual, Dentsu, Victor Entert...","Funimation, Bandai Entertainment",Sunrise,Original,25 min. per ep.,PG-13 - Teens 13 or older,2481.0,1467,94683,587,4300,46165,5121,5378,33719,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,"TV Tokyo, Dentsu",Unknown,Toei Animation,Manga,23 min. per ep.,PG - Children,3710.0,4369,13224,18,642,7314,766,1108,3394,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [49]:
anime = anime.replace('Unknown', np.nan)

In [50]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   MAL_ID         17562 non-null  int64 
 1   Name           17562 non-null  object
 2   Score          12421 non-null  object
 3   Genres         17499 non-null  object
 4   English name   6997 non-null   object
 5   Japanese name  17514 non-null  object
 6   Type           17525 non-null  object
 7   Episodes       17046 non-null  object
 8   Aired          17253 non-null  object
 9   Premiered      4745 non-null   object
 10  Producers      9768 non-null   object
 11  Licensors      3946 non-null   object
 12  Studios        10483 non-null  object
 13  Source         13995 non-null  object
 14  Duration       17007 non-null  object
 15  Rating         16874 non-null  object
 16  Ranked         15800 non-null  object
 17  Popularity     17562 non-null  int64 
 18  Members        17562 non-n

In [51]:
anime.isnull().sum()

MAL_ID               0
Name                 0
Score             5141
Genres              63
English name     10565
Japanese name       48
Type                37
Episodes           516
Aired              309
Premiered        12817
Producers         7794
Licensors        13616
Studios           7079
Source            3567
Duration           555
Rating             688
Ranked            1762
Popularity           0
Members              0
Favorites            0
Watching             0
Completed            0
On-Hold              0
Dropped              0
Plan to Watch        0
Score-10           437
Score-9           3167
Score-8           1371
Score-7            503
Score-6            511
Score-5            584
Score-4            977
Score-3           1307
Score-2           1597
Score-1            459
dtype: int64

In [52]:
nb_null_anime = anime.duplicated().sum()
print(nb_null_anime)

0


In [53]:
anime["MAL_ID"].duplicated().sum()

np.int64(0)

In [54]:
anime["Name"].duplicated().sum()

np.int64(4)

In [55]:
anime[anime["Name"].duplicated(keep=False)]

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,Producers,Licensors,Studios,Source,Duration,Rating,Ranked,Popularity,Members,Favorites,Watching,Completed,On-Hold,Dropped,Plan to Watch,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
12826,35102,Hinamatsuri,6.79,"Historical, Kids",NaN,ひなまつり,OVA,1,NaN,NaN,NaN,NaN,NaN,Other,19 min.,G - All Ages,4486.0,10044,1201,2,52,399,17,92,641,22.0,18.0,38.0,58.0,21.0,12.0,6.0,4.0,1.0,5.0
12955,35279,Youkoso! Ecolo Shima,NaN,Kids,NaN,ようこそ！エコロ島,OVA,1,NaN,NaN,NaN,NaN,NaN,Original,17 min.,G - All Ages,13024.0,16791,100,0,7,21,2,35,35,4.0,NaN,NaN,4.0,2.0,2.0,NaN,NaN,NaN,4.0
13540,36296,Hinamatsuri,8.21,"Comedy, Sci-Fi, Seinen, Slice of Life, Superna...",Hinamatsuri,ヒナまつり,TV,12,"Apr 6, 2018 to Jun 22, 2018",Spring 2018,"Media Factory, Magic Capsule, Nippon Columbia,...",Funimation,feel.,Manga,23 min. per ep.,PG-13 - Teens 13 or older,298.0,391,347326,3265,27757,209253,9457,7827,93032,21484.0,49894.0,63480.0,29442.0,8408.0,2895.0,813.0,267.0,124.0,168.0
15379,39143,Youkoso! Ecolo Shima,NaN,Kids,NaN,ようこそ！ エコロ島,OVA,1,NaN,NaN,NaN,NaN,NaN,Original,17 min.,G - All Ages,13025.0,17368,54,0,1,9,2,21,21,2.0,NaN,NaN,1.0,2.0,NaN,1.0,NaN,NaN,2.0
16195,40496,Maou Gakuin no Futekigousha: Shijou Saikyou no...,7.34,"Action, Demons, Magic, Fantasy, School",The Misfit of Demon King Academy,魔王学院の不適合者 ～史上最強の魔王の始祖、転生して子孫たちの学校へ通う～,TV,13,"Jul 4, 2020 to Sep 26, 2020",Summer 2020,"Aniplex, Kadokawa",Aniplex of America,SILVER LINK.,Light novel,23 min. per ep.,R - 17+ (violence & profanity),2177.0,377,356060,3057,48873,236529,4880,9563,56215,18424.0,23990.0,50133.0,58622.0,27479.0,12119.0,5087.0,2364.0,1209.0,835.0
17543,48417,Maou Gakuin no Futekigousha: Shijou Saikyou no...,NaN,"Magic, Fantasy, School",NaN,魔王学院の不適合者 ～史上最強の魔王の始祖、転生して子孫たちの学校へ通う～,TV,NaN,NaN,NaN,Aniplex,NaN,SILVER LINK.,Light novel,NaN,R - 17+ (violence & profanity),NaN,2597,41425,281,7,5,11,1,41401,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17544,48418,Maou Gakuin no Futekigousha: Shijou Saikyou no...,NaN,"Action, Demons, Magic, Fantasy, School",NaN,魔王学院の不適合者 ～史上最強の魔王の始祖、転生して子孫たちの学校へ通う～,TV,NaN,NaN,NaN,Aniplex,NaN,SILVER LINK.,Light novel,NaN,R - 17+ (violence & profanity),NaN,5731,7707,41,1,2,5,2,7697,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [56]:
anime['Score'] = pd.to_numeric(anime['Score'], errors='coerce')
anime['Episodes'] = pd.to_numeric(anime['Episodes'], errors='coerce')
anime['Ranked'] = pd.to_numeric(anime['Ranked'], errors='coerce')

In [57]:
score_cols = ['Score-1', 'Score-2', 'Score-3', 'Score-4', 'Score-5', 'Score-6', 'Score-7', 'Score-8', 'Score-9', 'Score-10']
anime[score_cols] = anime[score_cols].apply(pd.to_numeric, errors='coerce')

In [58]:
anime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17562 entries, 0 to 17561
Data columns (total 35 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MAL_ID         17562 non-null  int64  
 1   Name           17562 non-null  object 
 2   Score          12421 non-null  float64
 3   Genres         17499 non-null  object 
 4   English name   6997 non-null   object 
 5   Japanese name  17514 non-null  object 
 6   Type           17525 non-null  object 
 7   Episodes       17046 non-null  float64
 8   Aired          17253 non-null  object 
 9   Premiered      4745 non-null   object 
 10  Producers      9768 non-null   object 
 11  Licensors      3946 non-null   object 
 12  Studios        10483 non-null  object 
 13  Source         13995 non-null  object 
 14  Duration       17007 non-null  object 
 15  Rating         16874 non-null  object 
 16  Ranked         15800 non-null  float64
 17  Popularity     17562 non-null  int64  
 18  Member

In [59]:
anime['Type'].value_counts()

Type
TV         4996
OVA        3894
Movie      3041
Special    2218
ONA        1907
Music      1469
Name: count, dtype: int64

In [60]:
anime['Source'].value_counts()

Source
Original         5215
Manga            3825
Visual novel      993
Game              880
Light novel       768
Other             597
Novel             510
Music             317
4-koma manga      288
Web manga         252
Picture book      147
Book              112
Card game          64
Digital manga      15
Radio              12
Name: count, dtype: int64

In [61]:
anime['Rating'].value_counts()

Rating
PG-13 - Teens 13 or older         6132
G - All Ages                      5782
PG - Children                     1461
Rx - Hentai                       1345
R - 17+ (violence & profanity)    1157
R+ - Mild Nudity                   997
Name: count, dtype: int64

In [62]:
genres = anime['Genres'].dropna().str.split(', ').explode()
print(genres.value_counts().head(15))

Genres
Comedy           6029
Action           3888
Fantasy          3285
Adventure        2957
Kids             2665
Drama            2619
Sci-Fi           2583
Music            2244
Shounen          2003
Slice of Life    1914
Romance          1899
School           1642
Supernatural     1479
Hentai           1348
Historical       1144
Name: count, dtype: int64


In [63]:
studios = anime['Studios'].dropna().str.split(', ').explode()
print(studios.value_counts().head(15))

Studios
Toei Animation          778
Sunrise                 502
J.C.Staff               382
Madhouse                364
Production I.G          335
TMS Entertainment       300
Studio Deen             287
Studio Pierrot          263
OLM                     237
Nippon Animation        217
A-1 Pictures            209
Shin-Ei Animation       178
DLE                     175
Tatsunoko Production    170
Xebec                   155
Name: count, dtype: int64


## Synthèse anime

| Anomalie | Détail | Action |
|----------|--------|--------|
| "Unknown" comme sentinelle | Masquait les vraies valeurs manquantes (0 null affiché par pandas) | Remplacé par NaN via `replace('Unknown', np.nan)` |
| Colonnes numériques en string | Score, Episodes, Ranked, Score-1 à Score-10 stockés en object | Convertis en float via `pd.to_numeric(errors='coerce')` |
| 4 noms dupliqués | Hinamatsuri (OVA vs TV), Youkoso! Ecolo Shima, Maou Gakuin (saisons) | Légitimes : MAL_ID distincts, pas de suppression |
| Premiered | 12 817 NaN (73%) | Normal : seuls les animes TV saisonniers ont une saison de première |
| Licensors | 13 616 NaN (77%) | Normal : beaucoup d'animes n'ont pas de distributeur occidental |
| English name | 10 565 NaN (60%) | Beaucoup d'animes n'ont pas de titre anglais officiel |
| Genres et Studios | Valeurs multi-valuées séparées par ", " | Exploitables après split, pas de nettoyage nécessaire |

# 4 EDA, Nettoyage et Audit de ratings_complete

In [64]:
rating_complete.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57633278 entries, 0 to 57633277
Data columns (total 3 columns):
 #   Column    Dtype
---  ------    -----
 0   user_id   int64
 1   anime_id  int64
 2   rating    int64
dtypes: int64(3)
memory usage: 1.3 GB


In [65]:
rating_complete['rating'].value_counts()

rating
8     14642156
7     13325549
9      9773857
6      6849293
10     6716048
5      3436250
4      1455102
3       696048
2       405556
1       333419
Name: count, dtype: int64

In [66]:
rating_complete['rating'].describe()

count    5.763328e+07
mean     7.510789e+00
std      1.697722e+00
min      1.000000e+00
25%      7.000000e+00
50%      8.000000e+00
75%      9.000000e+00
max      1.000000e+01
Name: rating, dtype: float64

In [67]:
rating_complete.isnull().sum()

user_id     0
anime_id    0
rating      0
dtype: int64

In [68]:
rating_complete[['user_id', 'anime_id']].duplicated().sum()

np.int64(0)

In [69]:
rating_complete['user_id'].nunique()


310059

In [70]:
rating_complete['anime_id'].nunique()

16872

In [71]:
anime_ids = set(anime['MAL_ID'])
animelist_ids = set(animelist['anime_id'])
rating_ids = set(rating_complete['anime_id'])

print(f"anime_id dans animelist.csv mais pas dans anime.csv : {len(animelist_ids - anime_ids)}")
print(f"anime_id dans rating_complete.csv mais pas dans anime.csv : {len(rating_ids - anime_ids)}")

anime_id dans animelist.csv mais pas dans anime.csv : 0
anime_id dans rating_complete.csv mais pas dans anime.csv : 0


## Synthèse rating_complete

| Point | Détail |
|-------|--------|
| Nulls | 0 sur toutes les colonnes |
| Doublons | 0 (clé composite user_id + anime_id unique) |
| Users uniques | 310 059 |
| Animes uniques | 16 872 |
| Plage de notes | 1 à 10 (pas de 0 contrairement à animelist) |
| Distribution | Skew positif : médiane = 8, moyenne = 7.51 |
| Cohérence | 0 anime_id orphelins (tous présents dans anime.csv) |

**Ce fichier est propre et prêt à l'emploi.** Aucun nettoyage nécessaire.

# 5 EDA, Nettoyage et Audit de anime_with_synopsis

In [72]:
anime_with_synopsis.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16214 entries, 0 to 16213
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   MAL_ID     16214 non-null  int64 
 1   Name       16214 non-null  object
 2   Score      16214 non-null  object
 3   Genres     16214 non-null  object
 4   sypnopsis  16206 non-null  object
dtypes: int64(1), object(4)
memory usage: 633.5+ KB


In [73]:
anime_with_synopsis.isnull().sum()

MAL_ID       0
Name         0
Score        0
Genres       0
sypnopsis    8
dtype: int64

In [74]:
anime_with_synopsis = anime_with_synopsis.rename(columns={'sypnopsis': 'Synopsis'})

In [75]:
anime_with_synopsis.duplicated().sum()

np.int64(0)

In [76]:
anime_with_synopsis["MAL_ID"].duplicated().sum()

np.int64(0)

In [77]:
anime_with_synopsis['Synopsis'].str.len().describe()

count    16206.000000
mean       377.162471
std        340.194235
min         10.000000
25%        107.000000
50%        264.000000
75%        575.000000
max       3047.000000
Name: Synopsis, dtype: float64

In [78]:
(anime_with_synopsis['Synopsis'].str.len() < 50).sum()

np.int64(1375)

In [79]:
synopsis_ids = set(anime_with_synopsis['MAL_ID'])
anime_ids = set(anime['MAL_ID'])

print(f"Dans anime mais pas dans synopsis : {len(anime_ids - synopsis_ids)}")
print(f"Dans synopsis mais pas dans anime : {len(synopsis_ids - anime_ids)}")

Dans anime mais pas dans synopsis : 1348
Dans synopsis mais pas dans anime : 0


## Synthèse anime_with_synopsis

| Point | Détail |
|-------|--------|
| Colonne mal nommée | `sypnopsis` → renommée en `Synopsis` |
| Doublons | 0 (MAL_ID unique) |
| Couverture | 1348 animes dans anime.csv n'ont pas de synopsis |
| Orphelins | 0 (tous les MAL_ID de synopsis existent dans anime.csv) |
| Synopsis courts (< 50 chars) | À vérifier — potentiellement inutiles pour le full-text search |


# 6 Synthèse Intermèdiaire Et Prochaines Etapes


## Bilan qualité par fichier

| Fichier | Lignes | Qualité | Problèmes majeurs |
|---------|--------|---------|-------------------|
| animelist.csv | 109M | Moyenne | rating=0 (43%), watched_episodes=65535, watching_status aberrants |
| anime.csv | 17 562 | Faible | "Unknown" masquant les NaN, types incohérents, colonnes très incomplètes |
| rating_complete.csv | 57M | Bonne | Propre, aucun nettoyage nécessaire |
| anime_with_synopsis.csv | 16 214 | Bonne | Colonne mal nommée, 1348 animes sans synopsis |

## Actions de nettoyage à appliquer (Feature Engineering)

1. **animelist** : traiter les rating=0 (exclure ou flagger), corriger watched_episodes=65535 via croisement avec anime.csv
2. **anime** : conversions de types déjà faites, créer des features métier (score de popularité pondéré, ratio dropped/completed, classification studios)
3. **rating_complete** : prêt à l'emploi
4. **anime_with_synopsis** : merger avec anime.csv pour enrichir le dataset gold
5. **Export** : produire un dataset "gold" nettoyé et enrichi en CSV + JSON pour indexation dans Elasticsearch

# 7 Merge Dataset + Feature Engineering

In [80]:
anime_gold = anime.merge(anime_with_synopsis[['MAL_ID', 'Synopsis']], on='MAL_ID', how='left')

In [83]:

C = anime_gold['Score'].mean() 
m = anime_gold['Members'].quantile(0.25) 
anime_gold['weighted_score'] = ((anime_gold['Members'] * anime_gold['Score'] + m * C) / (anime_gold['Members'] + m))

In [84]:
studio_counts = anime_gold['Studios'].dropna().str.split(', ').explode().value_counts()
top_studios = set(studio_counts.head(20).index)
mid_studios = set(studio_counts.head(100).index) - top_studios

def classify_studio(s):
    if pd.isna(s):
        return 'unknown'
    main_studio = s.split(', ')[0]
    if main_studio in top_studios:
        return 'top'
    elif main_studio in mid_studios:
        return 'mid'
    return 'small'

anime_gold['studio_tier'] = anime_gold['Studios'].apply(classify_studio)
anime_gold['studio_tier'].value_counts()

studio_tier
unknown    7079
top        5095
mid        3341
small      2047
Name: count, dtype: int64

In [86]:
def parse_duration(df):
    if pd.isna(df):
        return np.nan
    hours = pd.Series(df).str.extract(r'(\d+)\s*hr')[0].values[0]
    mins = pd.Series(df).str.extract(r'(\d+)\s*min')[0].values[0]
    h = float(hours) if pd.notna(hours) else 0
    m = float(mins) if pd.notna(mins) else 0
    return h * 60 + m

anime_gold['duration_min'] = anime_gold['Duration'].apply(parse_duration)

In [87]:
anime_gold.to_csv('../data/anime_gold.csv', index=False)

In [88]:
anime_gold.to_json('../data/anime_gold.json', orient='records', force_ascii=False)